In [ ]:
import requests
import xml.etree.ElementTree as ET

headers = {"User-Agent": "Mozilla/5.0 (pesquisa academica - IC UFCAT)"}

candidatos = {
    "geral": "https://g1.globo.com/rss/g1/",
    "politica": "https://g1.globo.com/rss/g1/politica/",
    "saude": "https://g1.globo.com/rss/g1/saude/",
    "ciencia": "https://g1.globo.com/rss/g1/ciencia-e-saude/",
}

for nome, url in candidatos.items():
    try:
        r = requests.get(url, headers=headers, timeout=15)
        itens = ET.fromstring(r.content).findall(".//item") if r.status_code == 200 else []
        print(nome, "| status:", r.status_code, "|", len(itens), "itens")
    except Exception as e:
        print(nome, "| erro:", e)

geral | status: 200 | 100 itens
politica | status: 200 | 100 itens
saude | status: 200 | 100 itens
ciencia | status: 200 | 100 itens


In [ ]:
import requests
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup
import pandas as pd
import time

headers = {"User-Agent": "Mozilla/5.0 (pesquisa academica - IC UFCAT)"}

feeds = [
    "https://g1.globo.com/rss/g1/politica/",
    "https://g1.globo.com/rss/g1/saude/",
    "https://g1.globo.com/rss/g1/ciencia-e-saude/",
]
POR_FEED = 20

def pegar_itens(url_feed):
    r = requests.get(url_feed, headers=headers, timeout=15)
    itens = ET.fromstring(r.content).findall(".//item")
    return [{"url": i.findtext("link"), "data": i.findtext("pubDate")} for i in itens]

def pegar_texto(url):
    r = requests.get(url, headers=headers, timeout=15)
    soup = BeautifulSoup(r.text, "html.parser")
    titulo = soup.select_one("h1").get_text(strip=True)

    # tentativa 1: paragrafos de conteudo do G1
    ps = soup.select("p.content-text__container")
    # tentativa 2: todos os paragrafos dentro de <article>
    if not ps:
        art = soup.find("article")
        ps = art.find_all("p") if art else []
    # tentativa 3: qualquer paragrafo longo da pagina
    if not ps:
        ps = [p for p in soup.find_all("p") if len(p.get_text(strip=True)) > 60]

    texto = " ".join(p.get_text(" ", strip=True) for p in ps)
    return titulo, texto, len(ps)

noticias_g1 = []
descartadas = []

for feed in feeds:
    categoria = feed.rstrip("/").split("/")[-1]
    try:
        itens = pegar_itens(feed)[:POR_FEED]
    except Exception as e:
        print("Erro no feed", feed, e)
        continue
    for item in itens:
        try:
            titulo, texto, n_par = pegar_texto(item["url"])
            if len(texto) > 200:
                noticias_g1.append({
                    "titulo": titulo, "texto_completo": texto,
                    "data": item["data"], "categoria": categoria,
                    "url": item["url"], "label": 0,
                })
            else:
                descartadas.append((item["url"], n_par, len(texto)))
        except Exception as e:
            print("Erro em", item["url"], e)
        time.sleep(2)
    print(f"{categoria}: ok")

df_g1 = pd.DataFrame(noticias_g1)
df_g1.to_csv("g1_corpus.csv", index=False)
print("Coletadas:", df_g1.shape)
print("Descartadas:", len(descartadas))
for d in descartadas[:5]:
    print(d)
print(df_g1["texto_completo"].str.len().describe())

politica: ok
saude: ok
ciencia-e-saude: ok
Coletadas: (60, 6)
Descartadas: 0
count       60.000000
mean      3837.033333
std       2087.897783
min       1138.000000
25%       2498.250000
50%       3352.000000
75%       4376.250000
max      11710.000000
Name: texto_completo, dtype: float64


In [ ]:
import pandas as pd

df_boatos = pd.read_csv("boatos_corpus.csv")
df_g1 = pd.read_csv("g1_corpus.csv")

colunas = ["titulo", "texto_completo", "data", "categoria", "url", "label"]
corpus = pd.concat([df_boatos[colunas], df_g1[colunas]], ignore_index=True)
corpus.to_csv("corpus_atualizado_v0.csv", index=False)

print(corpus.shape)
print(corpus["label"].value_counts())
print(corpus.groupby("label")["texto_completo"].apply(lambda s: s.str.len().median()))

from google.colab import files
files.download("corpus_atualizado_v0.csv")

(110, 6)
label
0    60
1    50
Name: count, dtype: int64
label
0    3352.0
1    4590.5
Name: texto_completo, dtype: float64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import requests
import time
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup
import pandas as pd

headers = {"User-Agent": "Mozilla/5.0 (pesquisa academica - IC UFCAT)"}
corpus = pd.read_csv("corpus_atualizado_v0.csv")

# 1) fica so com o que e de 2026
corpus["dt"] = pd.to_datetime(corpus["data"], utc=True, format="mixed")
corpus = corpus[corpus["dt"].dt.year == 2026].drop(columns="dt")
print("Depois de remover as antigas:", corpus.shape)

def pegar_texto(url):
    r = requests.get(url, headers=headers, timeout=15)
    soup = BeautifulSoup(r.text, "html.parser")
    titulo = soup.select_one("h1").get_text(strip=True)
    ps = soup.select("p.content-text__container")
    if not ps:
        art = soup.find("article")
        ps = art.find_all("p") if art else []
    if not ps:
        ps = [p for p in soup.find_all("p") if len(p.get_text(strip=True)) > 60]
    return titulo, " ".join(p.get_text(" ", strip=True) for p in ps)

# 2) busca 20 novas, de 2026, que ainda nao estao no corpus
feeds = {
    "politica": "https://g1.globo.com/rss/g1/politica/",
    "geral": "https://g1.globo.com/rss/g1/",
}
ja_tem = set(corpus["url"])
novas = []

for categoria, url_feed in feeds.items():
    r = requests.get(url_feed, headers=headers, timeout=15)
    for item in ET.fromstring(r.content).findall(".//item"):
        if len(novas) >= 20:
            break
        link, data = item.findtext("link"), item.findtext("pubDate")
        if link in ja_tem or pd.to_datetime(data, utc=True).year != 2026:
            continue
        try:
            titulo, texto = pegar_texto(link)
            if len(texto) > 200:
                novas.append({"titulo": titulo, "texto_completo": texto,
                              "data": data, "categoria": categoria,
                              "url": link, "label": 0})
        except Exception as e:
            print("Erro em", link, e)
        time.sleep(2)

print("Novas coletadas:", len(novas))
corpus = pd.concat([corpus, pd.DataFrame(novas)], ignore_index=True)
corpus.to_csv("corpus_atualizado_v1.csv", index=False)

print(corpus.shape)
print(corpus["label"].value_counts())
print(corpus.groupby(["label", "categoria"]).size())

from google.colab import files
files.download("corpus_atualizado_v1.csv")

Depois de remover as antigas: (90, 6)
Novas coletadas: 20
(110, 6)
label
0    60
1    50
Name: count, dtype: int64
label  categoria     
0      politica          40
       saude             20
1      brasil             2
       eleicoes-2026     10
       entretenimento     4
       esporte            4
       politica          26
       religiao           2
       saude              1
       tecnologia         1
dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import re

corpus = pd.read_csv("corpus_atualizado_v1.csv")

# 1) tira as paginas de resultado de eleicao (nao sao boatos)
corpus = corpus[~((corpus["label"] == 1) & ~corpus["texto_completo"].str.startswith("Boato"))].copy()

# 2) limpa as marcas de checagem dos textos falsos
def limpar_falsa(t):
    t = re.sub(r"^Boato\s*[–-]\s*", "", t)
    t = t.replace("Análise ", "", 1)
    t = t.split("❌")[0]
    t = re.split(r"\s*Ps:\s*Esse artigo", t)[0]
    t = re.sub(r"\s*Fake news\s*$", "", t)
    t = re.sub(r"[^.]*Boatos\.org[^.]*\.\s*", "", t)
    return t.strip()

falsas = corpus["label"] == 1
corpus.loc[falsas, "texto_completo"] = corpus.loc[falsas, "texto_completo"].apply(limpar_falsa)

# 3) trunca todos os textos em 150 palavras
N = 150
corpus["texto_truncado"] = corpus["texto_completo"].apply(lambda t: " ".join(t.split()[:N]))

corpus.to_csv("corpus_atualizado_v2.csv", index=False)
print(corpus.shape)
print(corpus["label"].value_counts())
print(corpus[corpus["label"] == 1]["texto_truncado"].iloc[0][:300])

from google.colab import files
files.download("corpus_atualizado_v2.csv")

(100, 7)
label
0    60
1    40
Name: count, dtype: int64
A Coca-Cola fechou quatro fábricas no Brasil e demitiu cerca de 1.400 funcionários por causa do governo Lula e do PT. Publicações nas redes sociais passaram a compartilhar notícias sobre o fechamento de fábricas da Coca-Cola e a demissão de funcionários, acompanhadas de comentários críticos ao gover


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>